# DDA vs Bresenham Line Drawing Comparison

This notebook compares DDA and Bresenham line-drawing algorithms for:

- Horizontal lines
- Vertical lines
- Steep-slope lines

It tracks:

- Steps (loop iterations)
- Generated points
- Integer operations
- Floating-point operations


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
def dda_line(x1, y1, x2, y2):
    int_ops = 0
    float_ops = 0

    dx = x2 - x1
    dy = y2 - y1
    int_ops += 2

    steps = max(abs(dx), abs(dy))
    int_ops += 2

    points = []

    if steps == 0:
        points.append((x1, y1))
        return points, {
            "steps": 1,
            "points": 1,
            "integer_ops": int_ops,
            "float_ops": float_ops,
        }

    x_inc = dx / steps
    y_inc = dy / steps
    float_ops += 2

    x = float(x1)
    y = float(y1)

    for i in range(steps + 1):
        points.append((round(x), round(y)))
        if i < steps:
            x += x_inc
            y += y_inc
            float_ops += 2

    return points, {
        "steps": len(points),
        "points": len(points),
        "integer_ops": int_ops,
        "float_ops": float_ops,
    }


In [ ]:
def bresenham_line(x1, y1, x2, y2):
    int_ops = 0
    float_ops = 0

    dx_raw = x2 - x1
    dy_raw = y2 - y1
    int_ops += 2

    dx = abs(dx_raw)
    dy = abs(dy_raw)
    int_ops += 2

    sx = 1 if x1 < x2 else -1
    sy = 1 if y1 < y2 else -1

    err = dx - dy
    int_ops += 1

    x, y = x1, y1
    points = []

    while True:
        points.append((x, y))
        if x == x2 and y == y2:
            break

        e2 = 2 * err
        int_ops += 1

        if e2 > -dy:
            err -= dy
            x += sx
            int_ops += 2

        if e2 < dx:
            err += dx
            y += sy
            int_ops += 2

    return points, {
        "steps": len(points),
        "points": len(points),
        "integer_ops": int_ops,
        "float_ops": float_ops,
    }


In [ ]:
test_lines = {
    "Horizontal": (2, 3, 12, 3),
    "Vertical": (5, 1, 5, 12),
    "Steep slope": (2, 2, 6, 14),
}

comparison_rows = []
plot_results = {}

for line_type, coords in test_lines.items():
    x1, y1, x2, y2 = coords

    dda_points, dda_metrics = dda_line(x1, y1, x2, y2)
    bres_points, bres_metrics = bresenham_line(x1, y1, x2, y2)

    plot_results[line_type] = {
        "DDA": dda_points,
        "Bresenham": bres_points,
    }

    comparison_rows.append({
        "Line type": line_type,
        "Algorithm": "DDA",
        "Steps": dda_metrics["steps"],
        "Points": dda_metrics["points"],
        "Integer operations": dda_metrics["integer_ops"],
        "Floating-point operations": dda_metrics["float_ops"],
    })

    comparison_rows.append({
        "Line type": line_type,
        "Algorithm": "Bresenham",
        "Steps": bres_metrics["steps"],
        "Points": bres_metrics["points"],
        "Integer operations": bres_metrics["integer_ops"],
        "Floating-point operations": bres_metrics["float_ops"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
def plot_algorithm_points(ax, points, title, xlim, ylim):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]

    ax.scatter(xs, ys, marker="s", s=100)
    ax.plot(xs, ys, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.5)


for line_type, result in plot_results.items():
    dda_points = result["DDA"]
    bres_points = result["Bresenham"]

    all_x = [p[0] for p in dda_points + bres_points]
    all_y = [p[1] for p in dda_points + bres_points]

    xlim = (min(all_x) - 1, max(all_x) + 1)
    ylim = (min(all_y) - 1, max(all_y) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    fig.suptitle(f"{line_type} line", fontsize=14)

    plot_algorithm_points(axes[0], dda_points, "DDA", xlim, ylim)
    plot_algorithm_points(axes[1], bres_points, "Bresenham", xlim, ylim)

    plt.tight_layout()
    plt.show()


In [ ]:
summary_df = comparison_df.groupby("Algorithm")[["Steps", "Points", "Integer operations", "Floating-point operations"]].sum()
summary_df


## Summary

- Both algorithms generate the same line points for the tested horizontal, vertical, and steep-slope lines.
- DDA relies on floating-point division and incremental floating-point additions.
- Bresenham avoids floating-point math and uses integer-only updates.
- From the recorded operation counts, Bresenham is generally more efficient for integer-based raster displays.
